# 임베딩 기반 도구 검색: Claude를 수천 개의 도구로 확장하기

특화된 도구 수십 개로 Claude 애플리케이션을 만들다 보면 금세 한계에 부딪힙니다. 모든 도구 정의를 미리 제공하면 컨텍스트 윈도를 잡아먹고, 지연 시간과 비용이 늘어나며, Claude가 알맞은 도구를 찾기도 어려워집니다. 도구가 100개를 넘어가면 이 방식은 현실성이 없어집니다.

시맨틱 도구 검색은 도구를 "발견 가능한 리소스"로 다뤄 이 문제를 해결합니다. 수백 개의 정의를 앞단에 몰아넣는 대신, 필요할 때 관련 기능을 반환해 주는 `tool_search` 도구 하나만 Claude에 제공합니다. 컨텍스트 사용량을 90% 이상 줄이면서도 수천 개의 도구로 확장 가능한 애플리케이션을 만들 수 있습니다.

**이 쿡북을 마치면 다음을 할 수 있습니다.**
- 클라이언트 측 도구 검색을 구현해 Claude 애플리케이션을 수십 개에서 수천 개의 도구로 확장하기
- 시맨틱 임베딩으로 작업 맥락에 맞는 도구를 동적으로 발견하기
- 이 패턴을 도메인 특화 도구 라이브러리(API, 데이터베이스, 내부 시스템)에 적용하기

이 패턴은 컨텍스트 효율이 중요한 대규모 도구 생태계를 다루는 팀들이 실제 프로덕션에서 사용하고 있습니다. 설명을 명확히 하기 위해 여기서는 적은 수의 도구로 시연하지만, 같은 접근법이 수백, 수천 개 규모의 라이브러리로도 매끄럽게 확장됩니다.

## 사전 준비

이 가이드를 따라 하기 전에 다음을 확인하세요.

**필요한 사전 지식**
- Python 기초 — 함수, 딕셔너리, 기본 자료구조에 익숙할 것
- Claude 도구 사용에 대한 기본 이해 — 먼저 [도구 사용 가이드](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)를 읽어 보시길 권합니다

**필요한 도구**
- Python 3.11 이상
- Anthropic API 키 ([여기서 발급](https://docs.anthropic.com/claude/reference/getting-started-with-the-api))

## 준비

먼저 필요한 의존성을 설치합니다:

In [68]:
# Note: we use -q to avoid printing too much to stdout
# Use --only-binary to avoid build issues with pythran
%pip install --only-binary :all: -q anthropic sentence-transformers numpy python-dotenv

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


`.env` 파일에 다음 내용이 들어 있는지 확인하세요:
```
ANTHROPIC_API_KEY=your_key_here
```

환경 변수를 불러오고 클라이언트를 설정합니다:

In [69]:
import json
import random
from datetime import datetime, timedelta
from typing import Any

import anthropic
import numpy as np
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

# Load environment variables from .env file
load_dotenv()

# Define model constant for easy updates
MODEL = "claude-sonnet-4-6"

# Initialize Claude client (API key loaded from environment)
claude_client = anthropic.Anthropic()

# Load the SentenceTransformer model
# all-MiniLM-L6-v2 is a lightweight model with 384 dimensional embeddings
# It will be downloaded from HuggingFace on first use
print("Loading SentenceTransformer model...")
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("✓ Clients initialized successfully")

Loading SentenceTransformer model...
✓ Clients initialized successfully


## 도구 라이브러리 정의하기

시맨틱 검색을 구현하려면 먼저 검색 대상이 될 도구가 필요합니다. 날씨와 금융, 두 범주에 걸쳐 도구 8개로 이뤄진 라이브러리를 만들겠습니다.

실제 애플리케이션에서는 내부 API, 데이터베이스 작업, 서드파티 연동에 걸쳐 수백, 수천 개의 도구를 다룰 수도 있습니다. 시맨틱 검색 방식은 수정 없이도 그런 대규모 라이브러리로 확장됩니다. 여기서 적은 수를 쓰는 것은 순전히 설명을 명확히 하기 위해서입니다.

In [70]:
# Define our tool library with 2 domains
TOOL_LIBRARY = [
    # Weather Tools
    {
        "name": "get_weather",
        "description": "Get the current weather in a given location",
        "input_schema": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city and state, e.g. San Francisco, CA",
                },
                "unit": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"],
                    "description": "The unit of temperature",
                },
            },
            "required": ["location"],
        },
    },
    {
        "name": "get_forecast",
        "description": "Get the weather forecast for multiple days ahead",
        "input_schema": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city and state",
                },
                "days": {
                    "type": "number",
                    "description": "Number of days to forecast (1-10)",
                },
            },
            "required": ["location", "days"],
        },
    },
    {
        "name": "get_timezone",
        "description": "Get the current timezone and time for a location",
        "input_schema": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City name or timezone identifier",
                }
            },
            "required": ["location"],
        },
    },
    {
        "name": "get_air_quality",
        "description": "Get current air quality index and pollutant levels for a location",
        "input_schema": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City name or coordinates",
                }
            },
            "required": ["location"],
        },
    },
    # Finance Tools
    {
        "name": "get_stock_price",
        "description": "Get the current stock price and market data for a given ticker symbol",
        "input_schema": {
            "type": "object",
            "properties": {
                "ticker": {
                    "type": "string",
                    "description": "Stock ticker symbol (e.g., AAPL, GOOGL)",
                },
                "include_history": {
                    "type": "boolean",
                    "description": "Include historical data",
                },
            },
            "required": ["ticker"],
        },
    },
    {
        "name": "convert_currency",
        "description": "Convert an amount from one currency to another using current exchange rates",
        "input_schema": {
            "type": "object",
            "properties": {
                "amount": {
                    "type": "number",
                    "description": "Amount to convert",
                },
                "from_currency": {
                    "type": "string",
                    "description": "Source currency code (e.g., USD)",
                },
                "to_currency": {
                    "type": "string",
                    "description": "Target currency code (e.g., EUR)",
                },
            },
            "required": ["amount", "from_currency", "to_currency"],
        },
    },
    {
        "name": "calculate_compound_interest",
        "description": "Calculate compound interest for investments over time",
        "input_schema": {
            "type": "object",
            "properties": {
                "principal": {
                    "type": "number",
                    "description": "Initial investment amount",
                },
                "rate": {
                    "type": "number",
                    "description": "Annual interest rate (as percentage)",
                },
                "years": {"type": "number", "description": "Number of years"},
                "frequency": {
                    "type": "string",
                    "enum": ["daily", "monthly", "quarterly", "annually"],
                    "description": "Compounding frequency",
                },
            },
            "required": ["principal", "rate", "years"],
        },
    },
    {
        "name": "get_market_news",
        "description": "Get recent financial news and market updates for a specific company or sector",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Company name, ticker symbol, or sector",
                },
                "limit": {
                    "type": "number",
                    "description": "Maximum number of news articles to return",
                },
            },
            "required": ["query"],
        },
    },
]

print(f"✓ Defined {len(TOOL_LIBRARY)} tools in the library")

✓ Defined 8 tools in the library


## 도구 임베딩 만들기

시맨틱 검색은 키워드를 찾는 것이 아니라 텍스트의 *의미*를 비교하는 방식으로 동작합니다. 이를 위해 각 도구 정의를 의미를 담아내는 **임베딩 벡터**로 변환해야 합니다.

도구 정의는 이름, 설명, 파라미터를 담은 구조화된 JSON 객체이므로, 먼저 각 도구를 사람이 읽을 수 있는 텍스트 표현으로 바꾼 뒤 SentenceTransformer의 `all-MiniLM-L6-v2` 모델로 임베딩 벡터를 생성합니다.

이 모델을 고른 이유는 다음과 같습니다.
- **가볍고 빠릅니다**(더 큰 모델의 768차원 이상에 비해 384차원에 불과합니다)
- **로컬에서 실행**되어 API 호출이 필요 없습니다
- **도구 검색에는 충분합니다**(정확도를 높이려면 더 큰 모델로 실험해 볼 수 있습니다)

먼저 도구 정의를 검색 가능한 텍스트로 바꾸는 함수를 만들어 보겠습니다:

In [71]:
def tool_to_text(tool: dict[str, Any]) -> str:
    """
    Convert a tool definition into a text representation for embedding.
    Combines the tool name, description, and parameter information.
    """
    text_parts = [
        f"Tool: {tool['name']}",
        f"Description: {tool['description']}",
    ]

    # Add parameter information
    if "input_schema" in tool and "properties" in tool["input_schema"]:
        params = tool["input_schema"]["properties"]
        param_descriptions = []
        for param_name, param_info in params.items():
            param_desc = param_info.get("description", "")
            param_type = param_info.get("type", "")
            param_descriptions.append(f"{param_name} ({param_type}): {param_desc}")

        if param_descriptions:
            text_parts.append("Parameters: " + ", ".join(param_descriptions))

    return "\n".join(text_parts)


# Test with one tool
sample_text = tool_to_text(TOOL_LIBRARY[0])
print("Sample tool text representation:")
print(sample_text)

Sample tool text representation:
Tool: get_weather
Description: Get the current weather in a given location
Parameters: location (string): The city and state, e.g. San Francisco, CA, unit (string): The unit of temperature


이제 모든 도구에 대한 임베딩을 생성합니다:

In [72]:
# Create embeddings for all tools
print("Creating embeddings for all tools...")

tool_texts = [tool_to_text(tool) for tool in TOOL_LIBRARY]

# Embed all tools at once using SentenceTransformer
# The model returns normalized embeddings by default
tool_embeddings = embedding_model.encode(tool_texts, convert_to_numpy=True)

print(f"✓ Created embeddings with shape: {tool_embeddings.shape}")
print(f"  - {tool_embeddings.shape[0]} tools")
print(f"  - {tool_embeddings.shape[1]} dimensions per embedding")

Creating embeddings for all tools...
✓ Created embeddings with shape: (8, 384)
  - 8 tools
  - 384 dimensions per embedding


## 도구 검색 구현하기

도구를 벡터로 임베딩했으니 이제 시맨틱 검색을 구현할 수 있습니다. 두 텍스트의 의미가 비슷하면 임베딩 벡터도 벡터 공간에서 가까이 위치합니다. 이 "가까움"은 **코사인 유사도**로 측정합니다.

검색 과정은 다음과 같습니다.
1. **질의 임베딩**: Claude의 자연어 검색 요청을 도구와 동일한 벡터 공간으로 변환합니다
2. **유사도 계산**: 질의 벡터와 각 도구 벡터 사이의 코사인 유사도를 계산합니다
3. **정렬 후 반환**: 유사도 점수로 도구를 정렬해 상위 N개를 반환합니다

시맨틱 검색을 사용하면 Claude가 정확한 도구 이름 대신 "날씨를 확인해야 해"나 "투자 수익률을 계산해" 같은 자연어로 검색할 수 있습니다.

검색 함수를 구현하고 샘플 질의로 테스트해 보겠습니다:

In [73]:
def search_tools(query: str, top_k: int = 5) -> list[dict[str, Any]]:
    """
    Search for tools using semantic similarity.

    Args:
        query: Natural language description of what tool is needed
        top_k: Number of top tools to return

    Returns:
        List of tool definitions most relevant to the query
    """
    # Embed the query using SentenceTransformer
    query_embedding = embedding_model.encode(query, convert_to_numpy=True)

    # Calculate cosine similarity using dot product
    # SentenceTransformer returns normalized embeddings, so dot product = cosine similarity
    similarities = np.dot(tool_embeddings, query_embedding)

    # Get top k indices
    top_indices = np.argsort(similarities)[-top_k:][::-1]

    # Return the corresponding tools with their scores
    results = []
    for idx in top_indices:
        results.append({"tool": TOOL_LIBRARY[idx], "similarity_score": float(similarities[idx])})

    return results


# Test the search function
test_query = "I need to check the weather"
test_results = search_tools(test_query, top_k=3)

print(f"Search query: '{test_query}'\n")
print("Top 3 matching tools:")
for i, result in enumerate(test_results, 1):
    tool_name = result["tool"]["name"]
    score = result["similarity_score"]
    print(f"{i}. {tool_name} (similarity: {score:.3f})")

Search query: 'I need to check the weather'

Top 3 matching tools:
1. get_weather (similarity: 0.560)
2. get_forecast (similarity: 0.508)
3. get_air_quality (similarity: 0.401)


## tool_search 도구 정의하기

이제 Claude가 필요할 때 다른 도구를 발견할 수 있게 해 주는 **메타 도구**를 구현합니다. Claude는 갖고 있지 않은 기능이 필요하면 이 `tool_search` 도구로 검색하고, 결과로 도구 정의를 받아 곧바로 그 도구를 사용할 수 있습니다.

처음에 Claude에 제공하는 도구는 이것 하나뿐입니다:

In [74]:
# The tool_search tool definition
TOOL_SEARCH_DEFINITION = {
    "name": "tool_search",
    "description": "Search for available tools that can help with a task. Returns tool definitions for matching tools. Use this when you need a tool but don't have it available yet.",
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Natural language description of what kind of tool you need (e.g., 'weather information', 'currency conversion', 'stock prices')",
            },
            "top_k": {
                "type": "number",
                "description": "Number of tools to return (default: 5)",
            },
        },
        "required": ["query"],
    },
}

print("✓ Tool search definition created")

✓ Tool search definition created


이제 Claude의 `tool_search` 호출을 처리해 발견된 도구를 반환하는 핸들러를 구현합니다:

In [75]:
def handle_tool_search(query: str, top_k: int = 5) -> list[dict[str, Any]]:
    """
    Handle a tool_search invocation and return tool references.

    Returns a list of tool_reference content blocks for discovered tools.
    """
    # Search for relevant tools
    results = search_tools(query, top_k=top_k)

    # Create tool_reference objects instead of full definitions
    tool_references = [
        {"type": "tool_reference", "tool_name": result["tool"]["name"]} for result in results
    ]

    print(f"\n🔍 Tool search: '{query}'")
    print(f"   Found {len(tool_references)} tools:")
    for i, result in enumerate(results, 1):
        print(f"   {i}. {result['tool']['name']} (similarity: {result['similarity_score']:.3f})")

    return tool_references


# Test the handler
test_result = handle_tool_search("stock market data", top_k=3)
print(f"\nReturned {len(test_result)} tool references:")
for ref in test_result:
    print(f"  {ref}")


🔍 Tool search: 'stock market data'
   Found 3 tools:
   1. get_stock_price (similarity: 0.524)
   2. get_market_news (similarity: 0.469)
   3. calculate_compound_interest (similarity: 0.244)

Returned 3 tool references:
  {'type': 'tool_reference', 'tool_name': 'get_stock_price'}
  {'type': 'tool_reference', 'tool_name': 'get_market_news'}
  {'type': 'tool_reference', 'tool_name': 'calculate_compound_interest'}


## 목(mock) 도구 실행

이 시연에서는 도구 실행에 대한 목 응답을 만들겠습니다. 실제 애플리케이션에서는 이 부분이 실제 API나 서비스를 호출하게 됩니다:

In [76]:
def mock_tool_execution(tool_name: str, tool_input: dict[str, Any]) -> str:
    """
    Generate realistic mock responses for tool executions.

    Args:
        tool_name: Name of the tool being executed
        tool_input: Input parameters for the tool

    Returns:
        Mock response string appropriate for the tool
    """
    # Weather tools
    if tool_name == "get_weather":
        location = tool_input.get("location", "Unknown")
        unit = tool_input.get("unit", "fahrenheit")
        temp = random.randint(15, 30) if unit == "celsius" else random.randint(60, 85)
        conditions = random.choice(["sunny", "partly cloudy", "cloudy", "rainy"])
        return json.dumps(
            {
                "location": location,
                "temperature": temp,
                "unit": unit,
                "conditions": conditions,
                "humidity": random.randint(40, 80),
                "wind_speed": random.randint(5, 20),
            }
        )

    elif tool_name == "get_forecast":
        location = tool_input.get("location", "Unknown")
        days = int(tool_input.get("days", 5))
        forecast = []
        for i in range(days):
            date = (datetime.now() + timedelta(days=i)).strftime("%Y-%m-%d")
            forecast.append(
                {
                    "date": date,
                    "high": random.randint(20, 30),
                    "low": random.randint(10, 20),
                    "conditions": random.choice(["sunny", "cloudy", "rainy", "partly cloudy"]),
                }
            )
        return json.dumps({"location": location, "forecast": forecast})

    elif tool_name == "get_timezone":
        location = tool_input.get("location", "Unknown")
        return json.dumps(
            {
                "location": location,
                "timezone": "UTC+9",
                "current_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "utc_offset": "+09:00",
            }
        )

    elif tool_name == "get_air_quality":
        location = tool_input.get("location", "Unknown")
        aqi = random.randint(20, 150)
        categories = {
            (0, 50): "Good",
            (51, 100): "Moderate",
            (101, 150): "Unhealthy for Sensitive Groups",
        }
        category = next(cat for (low, high), cat in categories.items() if low <= aqi <= high)
        return json.dumps(
            {
                "location": location,
                "aqi": aqi,
                "category": category,
                "pollutants": {
                    "pm25": random.randint(5, 50),
                    "pm10": random.randint(10, 100),
                    "o3": random.randint(20, 80),
                },
            }
        )

    # Finance tools
    elif tool_name == "get_stock_price":
        ticker = tool_input.get("ticker", "UNKNOWN")
        return json.dumps(
            {
                "ticker": ticker,
                "price": round(random.uniform(100, 500), 2),
                "change": round(random.uniform(-5, 5), 2),
                "change_percent": round(random.uniform(-2, 2), 2),
                "volume": random.randint(1000000, 10000000),
                "market_cap": f"${random.randint(100, 1000)}B",
            }
        )

    elif tool_name == "convert_currency":
        amount = tool_input.get("amount", 0)
        from_currency = tool_input.get("from_currency", "USD")
        to_currency = tool_input.get("to_currency", "EUR")
        # Mock exchange rate
        rate = random.uniform(0.8, 1.2)
        converted = round(amount * rate, 2)
        return json.dumps(
            {
                "original_amount": amount,
                "from_currency": from_currency,
                "to_currency": to_currency,
                "exchange_rate": round(rate, 4),
                "converted_amount": converted,
            }
        )

    elif tool_name == "calculate_compound_interest":
        principal = tool_input.get("principal", 0)
        rate = tool_input.get("rate", 0)
        years = tool_input.get("years", 0)
        frequency = tool_input.get("frequency", "monthly")

        # Calculate compound interest
        n_map = {"daily": 365, "monthly": 12, "quarterly": 4, "annually": 1}
        n = n_map.get(frequency, 12)
        final_amount = principal * (1 + rate / 100 / n) ** (n * years)
        interest_earned = final_amount - principal

        return json.dumps(
            {
                "principal": principal,
                "rate": rate,
                "years": years,
                "compounding_frequency": frequency,
                "final_amount": round(final_amount, 2),
                "interest_earned": round(interest_earned, 2),
            }
        )

    elif tool_name == "get_market_news":
        query = tool_input.get("query", "")
        limit = tool_input.get("limit", 5)
        news = []
        for i in range(min(limit, 5)):
            news.append(
                {
                    "title": f"{query} - News Article {i + 1}",
                    "source": random.choice(
                        [
                            "Bloomberg",
                            "Reuters",
                            "Financial Times",
                            "Wall Street Journal",
                        ]
                    ),
                    "published": (datetime.now() - timedelta(hours=random.randint(1, 24))).strftime(
                        "%Y-%m-%d %H:%M"
                    ),
                    "summary": f"Latest developments regarding {query}...",
                }
            )
        return json.dumps({"query": query, "articles": news, "count": len(news)})

    # Default fallback
    else:
        return json.dumps(
            {
                "status": "executed",
                "tool": tool_name,
                "message": f"Tool {tool_name} executed successfully with input: {json.dumps(tool_input)}",
            }
        )


print("✓ Mock tool execution function created")

✓ Mock tool execution function created


## 대화 루프 구현하기

이제 전부 합쳐 보겠습니다! 도구 검색 워크플로 전체를 처리하는 대화 루프를 만듭니다.

**대화 흐름:**
1. Claude는 `tool_search` 도구만 사용할 수 있는 상태로 시작합니다
2. Claude가 `tool_search`를 호출하면 시맨틱 검색을 실행해 일치하는 도구 정의를 반환합니다
3. 그러면 Claude가 발견한 도구를 곧바로 사용할 수 있습니다
4. Claude가 발견한 도구를 호출하면 이를 실행합니다(이 데모에서는 목 응답을 사용합니다)
5. Claude가 최종 답변에 도달할 때까지 루프가 이어집니다

In [77]:
def run_tool_search_conversation(user_message: str, max_turns: int = 5) -> None:
    """
    Run a conversation with Claude using the tool search pattern.

    Args:
        user_message: The initial user message
        max_turns: Maximum number of conversation turns
    """
    print(f"\n{'=' * 80}")
    print(f"USER: {user_message}")
    print(f"{'=' * 80}\n")

    # Initialize conversation with only tool_search available
    messages = [{"role": "user", "content": user_message}]

    for turn in range(max_turns):
        print(f"\n--- Turn {turn + 1} ---")

        # Call Claude with current message history
        response = claude_client.messages.create(
            model=MODEL,
            max_tokens=1024,
            tools=TOOL_LIBRARY + [TOOL_SEARCH_DEFINITION],
            messages=messages,
            # IMPORTANT: This beta header enables tool definitions in tool results
            extra_headers={"anthropic-beta": "advanced-tool-use-2025-11-20"},
        )

        # Add assistant's response to messages
        messages.append({"role": "assistant", "content": response.content})

        # Check if we're done
        if response.stop_reason == "end_turn":
            print("\n✓ Conversation complete\n")
            # Print final response
            for block in response.content:
                if block.type == "text":
                    print(f"ASSISTANT: {block.text}")
            break

        # Handle tool uses
        if response.stop_reason == "tool_use":
            tool_results = []

            for block in response.content:
                if block.type == "text":
                    print(f"\nASSISTANT: {block.text}")

                elif block.type == "tool_use":
                    tool_name = block.name
                    tool_input = block.input
                    tool_use_id = block.id

                    print(f"\n🔧 Tool invocation: {tool_name}")
                    print(f"   Input: {json.dumps(tool_input, indent=2)}")

                    if tool_name == "tool_search":
                        # Handle tool search
                        query = tool_input["query"]
                        top_k = tool_input.get("top_k", 5)

                        # Get tool references
                        tool_references = handle_tool_search(query, top_k)

                        # Create tool result with tool_reference content blocks
                        tool_results.append(
                            {
                                "type": "tool_result",
                                "tool_use_id": tool_use_id,
                                "content": tool_references,
                            }
                        )
                    else:
                        # Execute the discovered tool with mock data
                        mock_result = mock_tool_execution(tool_name, tool_input)

                        # Print a preview of the result
                        if len(mock_result) > 150:
                            print(f"   ✅ Mock result: {mock_result[:150]}...")
                        else:
                            print(f"   ✅ Mock result: {mock_result}")

                        tool_results.append(
                            {
                                "type": "tool_result",
                                "tool_use_id": tool_use_id,
                                "content": mock_result,
                            }
                        )

            # Add tool results to messages
            if tool_results:
                messages.append({"role": "user", "content": tool_results})
        else:
            print(f"\nUnexpected stop reason: {response.stop_reason}")
            break

    print(f"\n{'=' * 80}\n")


print("✓ Conversation loop implemented")

✓ Conversation loop implemented


## 예제 1: 날씨 질의

간단한 날씨 질문으로 테스트해 보겠습니다. Claude는 다음과 같이 동작해야 합니다.
1. `tool_search`를 호출해 날씨 관련 도구를 찾습니다
2. 결과로 날씨 도구 정의를 받습니다
3. 발견한 도구 중 하나를 사용합니다

In [78]:
run_tool_search_conversation("What's the weather like in Tokyo?")


USER: What's the weather like in Tokyo?


--- Turn 1 ---

🔧 Tool invocation: get_weather
   Input: {
  "location": "Tokyo"
}
   ✅ Mock result: {"location": "Tokyo", "temperature": 75, "unit": "fahrenheit", "conditions": "partly cloudy", "humidity": 61, "wind_speed": 9}

--- Turn 2 ---

✓ Conversation complete

ASSISTANT: The weather in Tokyo is currently:
- **Temperature:** 75°F (about 24°C)
- **Conditions:** Partly cloudy
- **Humidity:** 61%
- **Wind Speed:** 9 mph

It's a pleasant day with comfortable temperatures and some cloud cover!




## 예제 2: 금융 질의

금융 도구를 발견해 사용해야 하는 금융 계산 질의를 시도해 보겠습니다:

In [79]:
run_tool_search_conversation(
    "If I invest $10,000 at 5% annual interest for 10 years with monthly compounding, how much will I have?"
)


USER: If I invest $10,000 at 5% annual interest for 10 years with monthly compounding, how much will I have?


--- Turn 1 ---

🔧 Tool invocation: calculate_compound_interest
   Input: {
  "principal": 10000,
  "rate": 5,
  "years": 10,
  "frequency": "monthly"
}
   ✅ Mock result: {"principal": 10000, "rate": 5, "years": 10, "compounding_frequency": "monthly", "final_amount": 16470.09, "interest_earned": 6470.09}

--- Turn 2 ---

✓ Conversation complete

ASSISTANT: If you invest $10,000 at 5% annual interest for 10 years with monthly compounding, you will have:

**Final Amount: $16,470.09**

This means you'll earn **$6,470.09** in interest over the 10-year period.

The monthly compounding means that interest is calculated and added to your principal every month, which allows your investment to grow faster than with annual compounding due to the effect of earning "interest on interest" more frequently.




## 마무리

이 쿡북에서는 Claude가 대규모 도구 라이브러리를 효율적으로 다룰 수 있게 해 주는 클라이언트 측 도구 검색 시스템을 구현했습니다. 다룬 내용은 다음과 같습니다.

- **시맨틱 도구 발견**: 임베딩으로 자연어 질의를 관련 도구에 매칭해, 사용 가능한 도구 전체를 미리 보지 않고도 Claude가 알맞은 기능을 찾도록 합니다
- **동적 도구 적재**: Claude의 도구 검색 기능을 사용해 도구 결과에 도구 정의를 담아 반환함으로써, Claude가 대화 도중에 새 도구를 발견해 즉시 사용할 수 있게 합니다
- **컨텍스트 최적화**: 초기 컨텍스트를 수천 토큰(도구 정의 19개 이상)에서 `tool_search` 정의 하나로 줄여 컨텍스트 사용량을 90% 이상 절감합니다

### 내 프로젝트에 적용하기

다음과 같은 경우에 도구 검색을 고려해 보세요.
- **특화된 도구가 20개를 넘고** 컨텍스트 사용량이 걱정되기 시작할 때
- 도구 라이브러리가 **시간이 지나며 계속 늘어나** 수작업 관리가 현실적이지 않을 때
- 엔드포인트가 수백 개인 **도메인 특화 API**(데이터베이스 작업, 내부 마이크로서비스, 서드파티 연동)를 지원해야 할 때
- **비용과 지연 시간 최적화**가 애플리케이션의 우선순위일 때

### 다음 단계

이 구현을 더 발전시키려면 다음을 시도해 보세요.

1. **임베딩 영속화**: 임베딩을 디스크에 캐싱해 세션마다 다시 계산하지 않도록 하면 시작 시간이 줄어듭니다
2. **검색 품질 개선**: 다른 임베딩 모델(예: `all-mpnet-base-v2` 같은 더 큰 모델)을 실험하거나, 시맨틱 검색과 키워드 매칭(BM25)을 결합한 하이브리드 검색을 구현해 보세요
3. **더 큰 라이브러리로 확장**: 도구 수백, 수천 개로 테스트해 프로덕션 규모에서 이 패턴이 어떻게 동작하는지 확인하세요
4. **도구 메타데이터 추가**: 사용 통계, 비용 정보, 신뢰도 점수 등을 검색 순위에 반영하세요
5. **캐싱 구현**: 자주 쓰이는 도구 정의를 캐싱해 반복 검색을 줄이세요

### 더 읽어 볼 자료

- [Claude 도구 사용 가이드](https://docs.anthropic.com/en/docs/build-with-claude/tool-use) — 도구로 무언가를 만들 때 참고할 종합 가이드
- [SentenceTransformers 문서](https://www.sbert.net/) — 임베딩 모델과 시맨틱 검색에 대해 더 알아보기
- [Tool Search Tool 문서](https://docs.anthropic.com/en/docs/build-with-claude/tool-use#tool-search) — 도구 검색 패턴에 대한 공식 문서